# Step 2 – Dynamic Data Test Guide

Notebook test lại Step 2 theo phiên bản hiện tại.

Luồng: `standardize_currency.py` → kiểm tra `review_bank.json` → `generate_dynamic_data.py` → validation cuối.

In [7]:
from pathlib import Path
import json, subprocess, sys
import pandas as pd

current = Path.cwd().resolve()
ROOT = None
for p in [current, *current.parents]:
    if (p / "README.md").exists() and (p / "data").exists() and (p / "src").exists():
        ROOT = p
        break
assert ROOT is not None, "Không tìm thấy repo root"
print("Repo root:", ROOT)
print("Python:", sys.executable)
print("Pandas:", pd.__version__)

Repo root: C:\Users\LucasNguyen\Desktop\NMPTDL&AI\Prac_T02\Prac_T02\data-analysis-ai-nhom11
Python: C:\Users\LucasNguyen\Desktop\NMPTDL&AI\Prac_T02\Prac_T02\data-analysis-ai-nhom11\.venv313\Scripts\python.exe
Pandas: 3.0.6


## 1. Kiểm tra script

In [8]:
SCRIPT_DIR = ROOT / "src" / "data_generation"
scripts = {
    "standardize_currency": SCRIPT_DIR / "standardize_currency.py",
    "generate_review_bank": SCRIPT_DIR / "generate_review_bank.py",
    "generate_dynamic_data": SCRIPT_DIR / "generate_dynamic_data.py",
}
for name, path in scripts.items():
    print(f"{name:25} ->", "OK" if path.exists() else "MISSING")
    assert path.exists(), f"Thiếu file: {path}"
print("SCRIPT CHECK PASSED")

standardize_currency      -> OK
generate_review_bank      -> OK
generate_dynamic_data     -> OK
SCRIPT CHECK PASSED


## 2. Chuẩn hóa tiền tệ

In [9]:
subprocess.run([sys.executable, str(scripts["standardize_currency"])], cwd=ROOT, check=True)
tesco_path = ROOT / "data" / "processed" / "tesco_products_vnd.csv"
tiki_path = ROOT / "data" / "processed" / "tiki_products_vnd.csv"
tesco = pd.read_csv(tesco_path)
tiki = pd.read_csv(tiki_path)
assert len(tesco) == 1200
assert len(tiki) == 5361
assert tesco["Discount_Price_VND"].isna().sum() == 3
assert tiki["Discount_Price_VND"].isna().sum() == 0
print("CURRENCY VALIDATION PASSED")

CURRENCY VALIDATION PASSED


## 3. Kiểm tra Review Bank

In [10]:
review_bank_path = ROOT / "data" / "processed" / "review_bank.json"
assert review_bank_path.exists(), "Thiếu review_bank.json"
with open(review_bank_path, "r", encoding="utf-8") as f:
    review_bank = json.load(f)
for rating in range(1, 6):
    key = str(rating)
    assert key in review_bank
    assert len(review_bank[key]) == 20
assert sum(len(v) for v in review_bank.values()) == 100
print("REVIEW BANK VALIDATION PASSED")

REVIEW BANK VALIDATION PASSED


## 4. Sinh Dynamic Data

In [11]:
subprocess.run([sys.executable, str(scripts["generate_dynamic_data"])], cwd=ROOT, check=True)
dynamic_path = ROOT / "data" / "processed" / "dynamic_transactions.csv"
assert dynamic_path.exists(), "Không tạo được dynamic_transactions.csv"
print("DYNAMIC DATA GENERATED")

DYNAMIC DATA GENERATED


## 5. Validation cuối Step 2

In [12]:
df = pd.read_csv(dynamic_path)
expected_columns = ["Transaction_ID","Customer_ID","Product_ID","Transaction_Date","Quantity","Unit_Price_VND","Revenue","Rating","Customer_Review","Gender","Age","City"]
assert df.shape == (10000, 12)
assert list(df.columns) == expected_columns
assert df["Transaction_ID"].nunique() == 10000
assert df.isna().sum().sum() == 0
assert df["Quantity"].between(1, 5).all()
assert df["Rating"].between(1, 5).all()
assert (df["Revenue"] == df["Unit_Price_VND"] * df["Quantity"]).all()
product_pool = pd.concat([tesco[["Product_ID","Discount_Price_VND"]], tiki[["Product_ID","Discount_Price_VND"]]], ignore_index=True)
product_pool["Product_ID"] = pd.to_numeric(product_pool["Product_ID"], errors="coerce")
product_pool["Discount_Price_VND"] = pd.to_numeric(product_pool["Discount_Price_VND"], errors="coerce")
product_pool = product_pool.dropna(subset=["Product_ID","Discount_Price_VND"]).copy()
product_pool["Product_ID"] = product_pool["Product_ID"].astype("int64")
product_pool = product_pool.drop_duplicates(subset=["Product_ID"], keep="first")
assert len(product_pool) == 6556
assert product_pool["Product_ID"].is_unique
assert df["Product_ID"].isin(product_pool["Product_ID"]).all()
for rating in range(1, 6):
    valid_reviews = set(review_bank[str(rating)])
    assert df.loc[df["Rating"] == rating, "Customer_Review"].isin(valid_reviews).all()
print("STEP 2 VALIDATION PASSED")
print("Transactions :", len(df))
print("Customers used:", df["Customer_ID"].nunique())
print("Products used :", df["Product_ID"].nunique())
print("\nRating distribution:")
print(df["Rating"].value_counts().sort_index())
df.head()

STEP 2 VALIDATION PASSED
Transactions : 10000
Customers used: 1988
Products used : 5099

Rating distribution:
Rating
1    1993
2    2024
3    1931
4    1977
5    2075
Name: count, dtype: int64


,Transaction_ID,Customer_ID,Product_ID,Transaction_Date,Quantity,Unit_Price_VND,Revenue,Rating,Customer_Review,Gender,Age,City
0,T000001,C01049,173798774,2026-05-27 07:53:23,3,37000,111000,4,"Với trải nghiệm này, tôi có trải nghiệm tích c...",Male,31,TP. Hồ Chí Minh
1,T000002,C00099,24659536,2025-11-03 05:40:50,3,23750,71250,3,"Với trải nghiệm này, sản phẩm dùng được, không...",Female,47,Vũng Tàu
2,T000003,C00503,16683725,2025-09-30 17:07:01,1,1010000,1010000,2,"Theo cảm nhận của tôi, sản phẩm vẫn dùng được ...",Male,23,Huế
3,T000004,C00810,570490,2026-07-05 23:45:05,2,585000,1170000,3,"Theo cảm nhận của tôi, sản phẩm đáp ứng nhu cầ...",Male,59,Hà Nội
4,T000005,C00241,325087836,2026-01-05 19:27:32,3,347232,1041696,2,Trải nghiệm chưa tốt và sản phẩm cần cải thiện...,Female,59,TP. Hồ Chí Minh


## Kết quả mong đợi

Nếu cell cuối in `STEP 2 VALIDATION PASSED` thì Step 2 hoạt động đúng.

Output chính: `data/processed/dynamic_transactions.csv`.